# FLN Level Populator — Table Extraction & Classification

**Three-Column Format: Q No. | Illustration | Question**

## Data Sources
| Document | Levels | Content |
|----------|--------|--------|
| `FLN_Formatted_Document_1.txt` | 1–22 | Explicit Q No. | Illustration | Question tables with actual row counts |
| `FLN_Formatted_Document_2.pdf` | 23–32 | Section structure with example questions; estimated image counts from sections |
| `Doc 2 (Place Value variant)` | 24 (alt) | Alternative version of Level 24 as Place Value (Tens & Ones) |

### Table Structure
```
Q No.  |  Illustration  |  Question
  1    |   [image]      |  [text]
  2    |   [image]      |  [text]
 ...   |   ...          |  ...
```

### Classification Logic
| Sub-Level | Type | Promotion Rule |
|-----------|------|----------------|
| X.0 | **Mastery** | Must pass to advance to X+1.0 |
| X.1 | **Easier Remediation** | Assigned if X.0 failed |
| X.2 | **Further Remediation** | Assigned if X.1 also failed |

### Image Counts

**Doc 1 (explicit table rows):**
- L1: 23+24 (no mastery)
- L2: 17+11+12 = 40
- L3: 70+15+19 = 104
- L11 (Review): ~5 per sub-level
- L17: 12 (further only)

**Doc 2 (from section structure):**
| Level | Mastery | Remed 1 | Remed 2 |
|-------|---------|---------|--------|
| L23 Review | 12 | 8 | 6 (default) |
| L24 Numbers 51-100 | 12 | 10 | 10 |
| L25 Carry Addition | 12 | 10 | 10 |
| L26 Borrow Subtraction | 12 | 10 | 10 |
| L27 Comparison | 12 | 10 | 10 |
| L28 Ordering | 12 | 10 | 10 |
| L29 Tally Marks | 10 | 8 | 8 |
| L30 Analog Clock | 12 | 10 | 8 |
| L31 Ordinal Positions | 10 | 8 | 8 |
| L32 Multiplication | 12 | 10 | 8 |

### Doc 2 Notes
- 110 pages covering Levels 23–32
- Each level: title → objective → X.0 description → X.1 → X.2
- 15 embedded worksheet images found in PDF (pages 5, 6, 7, 8, 9, 15, 16, 19, 20, 23, 24, 29)
- Level 24 has two variants: "Numbers 51-100" and "Place Value (Tens & Ones)"

**Total: 32 levels, 96 sub-levels, ~1,334 image/question rows**

In [ ]:
import csv
import json
import os
from pathlib import Path
from dataclasses import dataclass, asdict

BASE = Path("/kaggle/working/fln_output")

# ============================================================
# COMPLETE LEVEL MAP (1-32) extracted from the document
# ============================================================

LEVELS = [
    # (num, name, class_range, nipun_strand, is_review)
    (1,  "Quantity Comparison",          "1",   "Number Sense", False),
    (2,  "Odd One Out",                  "1",   "Classification", False),
    (3,  "Matching + Tracing Lines",     "1",   "Pre-Writing", False),
    (4,  "Numbers 1-10",                "1",   "Number Sense", False),
    (5,  "Finger Gesture Counting",      "1",   "Number Sense", False),
    (6,  "After, Between, Before",       "1",   "Number Sense", False),
    (7,  "Addition through Objects",     "1",   "Number Operations", False),
    (8,  "Subtraction (1-10)",           "1",   "Number Operations", False),
    (9,  "Pattern Recognition + Tracing","1",   "Patterns", False),
    (10, "Comparison - Numeral",         "1",   "Number Sense", False),
    (11, "Review Assessment 1",          "1",   "Review", True),
    (12, "Tens and Ones",                "1-2", "Place Value", False),
    (13, "Numbers 11-30",                "1-2", "Number Sense", False),
    (14, "Counting + Fun Trace",         "1-2", "Number Sense", False),
    (15, "After, Between, Before (11-30)","1-2","Number Sense", False),
    (16, "Addition (1-30)",              "1-2", "Number Operations", False),
    (17, "Subtraction (1-30)",           "1-2", "Number Operations", False),
    (18, "Ordering",                     "2",   "Number Sense", False),
    (19, "Numbers 31-50",                "2",   "Number Sense", False),
    (20, "Skip Counting in 2s/3s",       "2",   "Patterns", False),
    (21, "Comparison (1-50)",            "2",   "Number Sense", False),
    (22, "Ordering (51-100)",            "2",   "Number Sense", False),
    (23, "Review Assessment 3",          "2",   "Review", True),
    (24, "Place Value / Numbers 51-100", "2",   "Place Value", False),
    (25, "Carry Addition",               "2-3", "Number Operations", False),
    (26, "Borrow Subtraction",           "2-3", "Number Operations", False),
    (27, "Comparison (Up to 100)",       "2-3", "Number Sense", False),
    (28, "Ordering (Ascend/Descend)",    "2-3", "Number Sense", False),
    (29, "Data Handling - Tally Marks",  "3",   "Data Handling", False),
    (30, "Time - Analog Clock",          "3",   "Measurement", False),
    (31, "Ordinal Positions (1st-10th)", "3",   "Number Sense", False),
    (32, "Multiplication (Repeated Add)","3",   "Number Operations", False),
]

# ============================================================
# IMAGE COUNT PROFILES (from actual document tables)
# ============================================================

KNOWN_COUNTS = {
    # Doc 1: actual table row counts
    (2, 0): 17,   # L2.0 Mastery
    (2, 1): 11,   # L2.1 Easier
    (2, 2): 12,   # L2.2 Further
    (3, 0): 70,   # L3.0 Mastery
    (3, 1): 15,   # L3.1 Easier
    (3, 2): 19,   # L3.2 Further
    (17, 2): 12,  # L17.2 Further

    # Doc 2: estimated from section structure (FLN_Formatted_Document_2.pdf)
    (23, 0): 12,  # L23.0 Review Assessment — 4 sections
    (23, 1): 8,   # L23.1 Easier version
    (24, 0): 12,  # L24.0 Numbers 51-100
    (24, 1): 10,  # L24.1 Trace + small sequences
    (24, 2): 10,  # L24.2 Trace + fewer numbers
    (25, 0): 12,  # L25.0 Carry Addition — 2-digit+2-digit
    (25, 1): 10,  # L25.1 2-digit+1-digit
    (25, 2): 10,  # L25.2 Visual aids
    (26, 0): 12,  # L26.0 Borrow Subtraction — 2-digit-2-digit
    (26, 1): 10,  # L26.1 2-digit-1-digit
    (26, 2): 10,  # L26.2 Visual aids
    (27, 0): 12,  # L27.0 Comparison — 3 sections
    (27, 1): 10,  # L27.1 Reduced complexity
    (27, 2): 10,  # L27.2 Visual comparison
    (28, 0): 12,  # L28.0 Ordering — 4-5 numbers
    (28, 1): 10,  # L28.1 3-4 numbers
    (28, 2): 10,  # L28.2 Visual aids
    (29, 0): 10,  # L29.0 Tally Marks — 3 sections
    (29, 1): 8,   # L29.1 Simple tallies
    (29, 2): 8,   # L29.2 Match + guided
    (30, 0): 12,  # L30.0 Analog Clock — quarter-hour
    (30, 1): 10,  # L30.1 Half-hour
    (30, 2): 8,   # L30.2 Full-hour
    (31, 0): 10,  # L31.0 Ordinal Positions — write positions
    (31, 1): 8,   # L31.1 Circle/match positions
    (31, 2): 8,   # L31.2 Visual activities
    (32, 0): 12,  # L32.0 Multiplication — equal groups
    (32, 1): 10,  # L32.1 Count groups
    (32, 2): 8,   # L32.2 Visual grouping
}

def get_image_count(level_num, sub_num, is_review):
    key = (level_num, sub_num)
    if key in KNOWN_COUNTS:
        return KNOWN_COUNTS[key]
    if is_review:
        return {0: 8, 1: 6, 2: 6}[sub_num]
    if sub_num == 0:
        return 18
    elif sub_num == 1:
        return 14
    else:
        return 14

In [ ]:
# ============================================================
# CLASSIFICATION ENGINE
# ============================================================

@dataclass
class SubLevel:
    code: str
    level_num: int
    sub_num: int
    label: str
    is_mastery: bool
    num_images: int
    sections: list = None

@dataclass
class Level:
    number: int
    name: str
    class_range: str
    nipun_strand: str
    is_review: bool
    sub_levels: list = None

    def classify(self):
        self.sub_levels = []
        for s in range(3):
            mastery = (s == 0)
            labels = ["Mastery", "Easier_Remediation", "Further_Remediation"]
            n = get_image_count(self.number, s, self.is_review)
            sl = SubLevel(
                code=f"{self.number}.{s}",
                level_num=self.number,
                sub_num=s,
                label=labels[s],
                is_mastery=mastery,
                num_images=n,
                sections=[
                    (f"Section {chr(65+i)}", n // 2 if i == 0 else n - n // 2)
                    for i in range(2)
                ],
            )
            self.sub_levels.append(sl)

ALL_LEVELS = []
for num, name, cls, strand, review in LEVELS:
    lvl = Level(num, name, cls, strand, review)
    lvl.classify()
    ALL_LEVELS.append(lvl)

print(f"{'='*80}")
print(f"  FLN LEVEL MAP (1-{len(ALL_LEVELS)})")
print(f"{'='*80}")
print(f"{'Level':<8} {'Topic':<35} {'Class':<6} {'Strand':<20} {'Type':<10} {'SubLevels':<10} {'Images'}")
print(f"{'─'*8} {'─'*35} {'─'*6} {'─'*20} {'─'*10} {'─'*10} {'─'*8}")
total_images = 0
for lvl in ALL_LEVELS:
    ltype = "REVIEW" if lvl.is_review else "SKILL"
    img_sum = sum(sl.num_images for sl in lvl.sub_levels)
    total_images += img_sum
    sub_codes = ", ".join(sl.code for sl in lvl.sub_levels)
    print(f"{lvl.number:<8} {lvl.name:<35} {lvl.class_range:<6} {lvl.nipun_strand:<20} {ltype:<10} {sub_codes:<10} {img_sum}")
print(f"{'─'*80}")
print(f"  Total sub-levels: {sum(1 for l in ALL_LEVELS for _ in l.sub_levels)}")
print(f"  Mastery: {sum(1 for l in ALL_LEVELS for s in l.sub_levels if s.is_mastery)}")
print(f"  Remediation: {sum(1 for l in ALL_LEVELS for s in l.sub_levels if not s.is_mastery)}")
print(f"  Total images needed: {total_images}")

In [ ]:
# ============================================================
# TABLE ROW GENERATOR
# Format: Q No. | Illustration | Question
# ============================================================

def generate_table_rows(level_num, sub_num, count):
    rows = []
    for i in range(1, count + 1):
        rows.append({
            "q_no": i,
            "illustration": f"image_L{level_num}_{sub_num}_{i}.png",
            "question": "",
            "level": level_num,
            "sub_level": f"{level_num}.{sub_num}",
            "classification": ["mastery", "remediation_easier", "remediation_further"][sub_num],
        })
    return rows

all_rows = []
for lvl in ALL_LEVELS:
    for sl in lvl.sub_levels:
        rows = generate_table_rows(lvl.number, sl.sub_num, sl.num_images)
        all_rows.extend(rows)

print(f"Generated {len(all_rows)} total table rows")
print(f"\nSample (first 5 rows):")
for r in all_rows[:5]:
    print(f"  Q{r['q_no']:>3} | {r['illustration']:<30} | level={r['level']} sub={r['sub_level']} [{r['classification']}]")

In [ ]:
# ============================================================
# EXPORT: Classification Report & Table CSV
# ============================================================

def export_all():
    os.makedirs(BASE, exist_ok=True)

    with open(BASE / "fln_classification.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["level","name","sub_code","sub_num","label","is_mastery",
                     "class_range","nipun_strand","is_review","num_images"])
        for lvl in ALL_LEVELS:
            for sl in lvl.sub_levels:
                w.writerow([lvl.number, lvl.name, sl.code, sl.sub_num, sl.label,
                            sl.is_mastery, lvl.class_range, lvl.nipun_strand,
                            lvl.is_review, sl.num_images])

    with open(BASE / "fln_table_rows.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["q_no","illustration","question","level","sub_level","classification"])
        for r in all_rows:
            w.writerow([r["q_no"], r["illustration"], r["question"],
                        r["level"], r["sub_level"], r["classification"]])

    meta = []
    for lvl in ALL_LEVELS:
        meta.append({
            "level": lvl.number,
            "name": lvl.name,
            "class_range": lvl.class_range,
            "nipun_strand": lvl.nipun_strand,
            "is_review": lvl.is_review,
            "sub_levels": [asdict(sl) for sl in lvl.sub_levels],
            "total_images": sum(sl.num_images for sl in lvl.sub_levels),
        })
    with open(BASE / "fln_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Exported to {BASE}/:")
    print(f"  fln_classification.csv — {len(ALL_LEVELS)*3} sub-level rows")
    print(f"  fln_table_rows.csv — {len(all_rows)} question/image rows")
    print(f"  fln_metadata.json — level hierarchy")

export_all()

In [ ]:
# ============================================================
# DIRECTORY STRUCTURE GENERATOR (Pinterest format)
# ============================================================

def create_structure():
    for lvl in ALL_LEVELS:
        safe = lvl.name.replace(" ", "_").replace(",", "").replace("-","_")
        pad = f"{lvl.number:02d}"
        level_dir = BASE / "pinterest_images" / f"Level_{pad}_{safe}"

        for sl in lvl.sub_levels:
            sub_dir = level_dir / f"{lvl.number}.{sl.sub_num}_{sl.label}"
            os.makedirs(sub_dir, exist_ok=True)
            for i in range(1, min(sl.num_images + 1, 5)):
                (sub_dir / f"page_{i}.png").touch()

        print(f"  OK L{pad} {lvl.name:<35} > {len(lvl.sub_levels)} sub-dirs")

print("Creating Pinterest-style directory structure...")
print(f"{'='*60}")
create_structure()
print(f"\nDone. Replace placeholder PNGs with actual Pinterest images.")
print(f"\nTo use on Kaggle:")
print(f"  1. Upload Pinterest images to /kaggle/input/fln-images/")
print(f"  2. Run: !cp -r /kaggle/input/fln-images/* /kaggle/working/fln_output/pinterest_images/")
print(f"  3. Download fln_output/ as ZIP")

In [ ]:
# ============================================================
# VERIFICATION: Known document values
# ============================================================

print("Verification against known document values:")
print(f"{'Level':<8} {'Known':<8} {'Generated':<10} {'Match?'}")
print(f"{'─'*8} {'─'*8} {'─'*10} {'─'*8}")
for (l, s), known in sorted(KNOWN_COUNTS.items()):
    gen = get_image_count(l, s, False)
    match = "OK" if known == gen else f"DIFF (delta={known-gen})"
    print(f"{l}.{s:<6} {known:<8} {gen:<10} {match}")

print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
print(f"  Total levels:         {len(ALL_LEVELS)}")
print(f"  Total sub-levels:     {len(ALL_LEVELS) * 3}")
m = sum(1 for l in ALL_LEVELS for s in l.sub_levels if s.is_mastery)
r = sum(1 for l in ALL_LEVELS for s in l.sub_levels if not s.is_mastery)
print(f"  Mastery sub-levels:   {m}")
print(f"  Remediation sub-lvls: {r}")
print(f"  Total images/tables:  {len(all_rows)}")
print(f"{'='*60}")

---
## How It Works

### The Actual Table Format (from FLN_Formatted_Document_1.txt)
```
Q No.  |  Illustration           |  Question
-------+-------------------------+----------------------
  1    |  [pinterest image]      |  "Match equal groups"
  2    |  [pinterest image]      |  "Circle more/less"
  3    |  [pinterest image]      |  ...
```

### Classification Rules
| Field | Logic |
|-------|-------|
| `.0_Mastery` | Main level -- student must pass to advance |
| `.1_Easier_Remediation` | Same concept, simplified -- assigned after X.0 failure |
| `.2_Further_Remediation` | Maximum scaffolding -- assigned if X.1 also fails |

### Adaptive Progression
```
X.0 (Mastery) --pass--> X+1.0
   | fail
   v
X.1 (Easier) --pass--> X.0 (reattempt)
   | fail
   v
X.2 (Further) --pass--> X.0 (reattempt)
   | fail
   v
Teacher Intervention
```

### Image Counts (averages from document tables)
| Level Type | Mastery | Easier Remediation | Further Remediation |
|------------|---------|--------------------|--------------------|
| Skill Level | 17-70 | 11-23 | 12-24 |
| Review Assessment | 5-8 | 5-6 | 5-6 |

### To Use on Kaggle:
1. Upload this notebook
2. Settings > Accelerator > None (CPU is fine)
3. Run all cells > generates CSVs + directory structure
4. Copy real Pinterest images into respective sub-directories
5. Download `fln_output/` as ZIP